# Lightweight Repo-Aware Coding Agent (OpenAI + Tools + Skills)

This notebook builds a **small but practical coding agent** that can:

- inspect a repository
- choose tools dynamically
- edit files
- run shell checks
- verify outcomes
- keep structured execution trace
- integrate MCP-backed browser tools through adapters

The control loop blends three phases:

1. **Gather context**
2. **Take action**
3. **Verify results**

These are *not* rigid stages. The agent can switch phases at each step based on tool outputs and verification signals.


## Design goals

- Minimal dependencies (`openai` + Python stdlib)
- Explicit, readable code
- Modular `Tool` and `Skill` abstractions with registries
- Structured state for easier debugging and future packaging
- Bounded loop with safe stop conditions
- MCP browser integration via adapter pattern


In [ ]:
# If needed, install dependency once:
# %pip install openai


In [ ]:
from __future__ import annotations

import json
import os
import re
import shlex
import subprocess
import time
import traceback
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional

try:
    from openai import OpenAI
except Exception:
    OpenAI = None


def now_ts() -> float:
    return time.time()


def compact(obj: Any) -> str:
    return json.dumps(obj, indent=2, ensure_ascii=False)


## Core abstractions: Tool, Skill, registries, agent state

In [ ]:
@dataclass
class ToolResult:
    ok: bool
    content: str
    data: Dict[str, Any] = field(default_factory=dict)


@dataclass
class ToolCall:
    step: int
    phase: str
    tool_name: str
    args: Dict[str, Any]
    ok: bool
    content_preview: str
    timestamp: float


@dataclass
class AgentState:
    task: str
    repo_root: str
    step: int = 0
    done: bool = False
    phase: str = "gather_context"
    summary: str = ""
    scratchpad: Dict[str, Any] = field(default_factory=dict)
    messages: List[Dict[str, str]] = field(default_factory=list)
    trace: List[ToolCall] = field(default_factory=list)
    consecutive_errors: int = 0
    unchanged_iterations: int = 0


@dataclass
class Skill:
    name: str
    description: str
    system_hint: str


class SkillRegistry:
    def __init__(self):
        self._skills: Dict[str, Skill] = {}

    def register(self, skill: Skill):
        self._skills[skill.name] = skill

    def list(self) -> List[Skill]:
        return list(self._skills.values())


class Tool:
    name: str = "tool"
    description: str = ""

    def run(self, state: AgentState, **kwargs) -> ToolResult:
        raise NotImplementedError


class ToolRegistry:
    def __init__(self):
        self._tools: Dict[str, Tool] = {}

    def register(self, tool: Tool):
        self._tools[tool.name] = tool

    def get(self, name: str) -> Tool:
        return self._tools[name]

    def specs(self) -> List[Dict[str, Any]]:
        return [{"name": t.name, "description": t.description} for t in self._tools.values()]

    def names(self) -> List[str]:
        return list(self._tools.keys())


## Built-in tools

In [ ]:
class ListDirTool(Tool):
    name = "list_dir"
    description = "List files/directories at a path relative to repo_root. args: {path='.', max_entries=200}."

    def run(self, state: AgentState, path: str = ".", max_entries: int = 200, **kwargs) -> ToolResult:
        base = Path(state.repo_root)
        target = (base / path).resolve()
        if not str(target).startswith(str(base.resolve())):
            return ToolResult(False, "Path escapes repo root")
        if not target.exists():
            return ToolResult(False, f"Path not found: {target}")
        entries = sorted([p.name for p in target.iterdir()])[:max_entries]
        return ToolResult(True, "\n".join(entries), {"count": len(entries), "path": str(target)})


class ReadFileTool(Tool):
    name = "read_file"
    description = "Read text file. args: {path, start_line=1, end_line=400}."

    def run(self, state: AgentState, path: str, start_line: int = 1, end_line: int = 400, **kwargs) -> ToolResult:
        base = Path(state.repo_root)
        target = (base / path).resolve()
        if not str(target).startswith(str(base.resolve())):
            return ToolResult(False, "Path escapes repo root")
        if not target.exists() or not target.is_file():
            return ToolResult(False, f"File not found: {path}")
        text = target.read_text(encoding="utf-8")
        lines = text.splitlines()
        lo = max(1, start_line)
        hi = min(len(lines), end_line)
        snippet = "\n".join(f"{i+1:>4} | {lines[i]}" for i in range(lo-1, hi))
        return ToolResult(True, snippet, {"line_count": len(lines)})


class WriteFileTool(Tool):
    name = "write_file"
    description = "Write full text to file. args: {path, content, create_dirs=true}."

    def run(self, state: AgentState, path: str, content: str, create_dirs: bool = True, **kwargs) -> ToolResult:
        base = Path(state.repo_root)
        target = (base / path).resolve()
        if not str(target).startswith(str(base.resolve())):
            return ToolResult(False, "Path escapes repo root")
        if create_dirs:
            target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return ToolResult(True, f"Wrote {len(content)} chars to {path}")


class SearchTool(Tool):
    name = "search"
    description = "Regex search in repo text files. args: {pattern, path='.', max_hits=200}."

    def run(self, state: AgentState, pattern: str, path: str = ".", max_hits: int = 200, **kwargs) -> ToolResult:
        base = Path(state.repo_root)
        root = (base / path).resolve()
        if not str(root).startswith(str(base.resolve())):
            return ToolResult(False, "Path escapes repo root")
        rx = re.compile(pattern)
        hits = []
        for p in root.rglob("*"):
            if len(hits) >= max_hits:
                break
            if p.is_file() and ".git" not in p.parts:
                try:
                    txt = p.read_text(encoding="utf-8")
                except Exception:
                    continue
                for i, line in enumerate(txt.splitlines(), start=1):
                    if rx.search(line):
                        rel = p.relative_to(base)
                        hits.append(f"{rel}:{i}: {line[:180]}")
                        if len(hits) >= max_hits:
                            break
        return ToolResult(True, "\n".join(hits) if hits else "No matches", {"hits": len(hits)})


class ShellTool(Tool):
    name = "shell"
    description = "Run shell command at repo root. args: {cmd, timeout=20}."

    def run(self, state: AgentState, cmd: str, timeout: int = 20, **kwargs) -> ToolResult:
        try:
            proc = subprocess.run(cmd, shell=True, cwd=state.repo_root, text=True, capture_output=True, timeout=timeout)
            out = (proc.stdout or "") + ("\n" + proc.stderr if proc.stderr else "")
            return ToolResult(proc.returncode == 0, out.strip(), {"returncode": proc.returncode})
        except subprocess.TimeoutExpired:
            return ToolResult(False, f"Command timed out after {timeout}s")


class GitDiffTool(Tool):
    name = "git_diff"
    description = "Show git diff for working tree. args: {target=''} where target can be path/spec."

    def run(self, state: AgentState, target: str = "", **kwargs) -> ToolResult:
        cmd = f"git diff -- {shlex.quote(target)}" if target else "git diff"
        proc = subprocess.run(cmd, shell=True, cwd=state.repo_root, text=True, capture_output=True)
        if proc.returncode != 0:
            return ToolResult(False, proc.stderr.strip() or "git diff failed")
        return ToolResult(True, proc.stdout.strip() or "<no diff>")


class ScratchpadTool(Tool):
    name = "scratchpad"
    description = "Scratchpad memory ops. args: {op: set|get|append|keys, key?, value?}."

    def run(self, state: AgentState, op: str, key: Optional[str] = None, value: Any = None, **kwargs) -> ToolResult:
        if op == "set" and key is not None:
            state.scratchpad[key] = value
            return ToolResult(True, f"set {key}")
        if op == "get" and key is not None:
            return ToolResult(True, compact(state.scratchpad.get(key)))
        if op == "append" and key is not None:
            current = state.scratchpad.get(key, [])
            if not isinstance(current, list):
                return ToolResult(False, f"{key} is not a list")
            current.append(value)
            state.scratchpad[key] = current
            return ToolResult(True, f"appended to {key}")
        if op == "keys":
            return ToolResult(True, "\n".join(state.scratchpad.keys()))
        return ToolResult(False, "Invalid scratchpad op")


class VerifyTool(Tool):
    name = "verify"
    description = "Verification helper. args: {mode: command|file_contains|exists, ...}."

    def run(self, state: AgentState, mode: str, **kwargs) -> ToolResult:
        base = Path(state.repo_root)
        if mode == "exists":
            path = kwargs.get("path", "")
            ok = (base / path).exists()
            return ToolResult(ok, f"exists({path})={ok}")
        if mode == "file_contains":
            path = kwargs.get("path", "")
            needle = kwargs.get("needle", "")
            target = (base / path)
            if not target.exists():
                return ToolResult(False, f"Missing file: {path}")
            txt = target.read_text(encoding="utf-8")
            ok = needle in txt
            return ToolResult(ok, f"contains={ok}")
        if mode == "command":
            cmd = kwargs.get("cmd", "")
            timeout = int(kwargs.get("timeout", 30))
            return ShellTool().run(state, cmd=cmd, timeout=timeout)
        return ToolResult(False, f"Unknown verify mode: {mode}")


## MCP adapter pattern (for browser-backed tools)

In [ ]:
class MCPToolAdapter:
    """Adapter interface for MCP-backed tools.

    You can implement this against:
    - microsoft/playwright-mcp
    - ChromeDevTools/chrome-devtools-mcp
    """

    provider: str = "mcp"

    def call(self, method: str, params: Dict[str, Any]) -> Dict[str, Any]:
        raise NotImplementedError


class StubMCPAdapter(MCPToolAdapter):
    provider = "stub"

    def call(self, method: str, params: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "ok": False,
            "method": method,
            "params": params,
            "message": "No MCP client configured. Plug in a real adapter.",
        }


class BrowserNavigateTool(Tool):
    name = "browser_navigate"
    description = "Navigate URL via MCP adapter. args: {url}."

    def __init__(self, adapter: MCPToolAdapter):
        self.adapter = adapter

    def run(self, state: AgentState, url: str, **kwargs) -> ToolResult:
        resp = self.adapter.call("browser.navigate", {"url": url})
        return ToolResult(bool(resp.get("ok", False)), compact(resp), resp)


### MCP implementation guidance

To wire a real MCP adapter later:

1. Start MCP server process (`playwright-mcp` or `chrome-devtools-mcp`).
2. Implement `MCPToolAdapter.call()` to send JSON-RPC / SDK calls.
3. Register more browser tools (`browser_click`, `browser_snapshot`, etc.) that delegate to adapter methods.
4. Keep browser tools optional so the agent still works repo-locally without MCP.


## OpenAI reasoner + dynamic loop planner

In [ ]:
PLANNER_SYSTEM_PROMPT = """You are a coding-agent planner.
Choose one next step at a time for a repo-aware agent.
Blend phases dynamically: gather_context, take_action, verify_results.
Output STRICT JSON with keys:
- thought: short reasoning
- phase: one of gather_context|take_action|verify_results
- tool_name: tool name to call OR null if final
- tool_args: object
- done: boolean
- completion_note: string
Rules:
- Prefer small safe actions.
- Use verify_results before done=true.
- If task is complete and verified, set done=true and tool_name=null.
"""


class OpenAIReasoner:
    def __init__(self, model: str = "gpt-4.1-mini", api_key: Optional[str] = None):
        self.model = model
        self.api_key = api_key or os.getenv("OPENAI_API_KEY")
        if OpenAI is None:
            raise RuntimeError("openai package not installed. Install with: pip install openai")
        if not self.api_key:
            raise RuntimeError("OPENAI_API_KEY not set")
        self.client = OpenAI(api_key=self.api_key)

    def next_step(self, state: AgentState, tool_specs: List[Dict[str, Any]]) -> Dict[str, Any]:
        context = {
            "task": state.task,
            "step": state.step,
            "phase": state.phase,
            "summary": state.summary,
            "scratchpad": state.scratchpad,
            "recent_trace": [asdict(t) for t in state.trace[-5:]],
            "tools": tool_specs,
        }
        resp = self.client.responses.create(
            model=self.model,
            input=[
                {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
                {"role": "user", "content": compact(context)},
            ],
            temperature=0.2,
        )
        text = (getattr(resp, "output_text", "") or "").strip()
        if not text:
            raise RuntimeError("Empty model response")
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            m = re.search(r"\{.*\}", text, flags=re.S)
            if not m:
                raise
            return json.loads(m.group(0))


## Agent engine

In [ ]:
class RepoCodingAgent:
    def __init__(self, repo_root: str, reasoner: OpenAIReasoner, tools: ToolRegistry, skills: Optional[SkillRegistry] = None,
                 max_steps: int = 20, max_consecutive_errors: int = 3, max_unchanged_iterations: int = 6):
        self.repo_root = str(Path(repo_root).resolve())
        self.reasoner = reasoner
        self.tools = tools
        self.skills = skills or SkillRegistry()
        self.max_steps = max_steps
        self.max_consecutive_errors = max_consecutive_errors
        self.max_unchanged_iterations = max_unchanged_iterations

    def run(self, task: str) -> AgentState:
        state = AgentState(task=task, repo_root=self.repo_root)
        for i in range(1, self.max_steps + 1):
            state.step = i
            if state.done:
                break
            if state.consecutive_errors >= self.max_consecutive_errors:
                state.summary += "\nStopped: too many consecutive errors."
                break
            if state.unchanged_iterations >= self.max_unchanged_iterations:
                state.summary += "\nStopped: no progress detected."
                break
            try:
                plan = self.reasoner.next_step(state, self.tools.specs())
                phase = plan.get("phase", "gather_context")
                state.phase = phase
                thought = plan.get("thought", "")
                state.summary = (state.summary + f"\nStep {i} [{phase}]: {thought}").strip()

                if bool(plan.get("done", False)):
                    state.done = True
                    state.summary += f"\nDone: {plan.get('completion_note', 'Task marked complete by reasoner.')}"
                    break

                tool_name = plan.get("tool_name")
                tool_args = plan.get("tool_args") or {}
                if tool_name not in self.tools.names():
                    state.consecutive_errors += 1
                    state.summary += f"\nInvalid tool: {tool_name}"
                    continue

                result = self.tools.get(tool_name).run(state, **tool_args)
                state.trace.append(ToolCall(
                    step=i,
                    phase=phase,
                    tool_name=tool_name,
                    args=tool_args,
                    ok=result.ok,
                    content_preview=result.content[:500],
                    timestamp=now_ts(),
                ))
                if result.ok:
                    state.consecutive_errors = 0
                    state.unchanged_iterations = 0 if result.content.strip() else state.unchanged_iterations + 1
                else:
                    state.consecutive_errors += 1
                    state.unchanged_iterations += 1

                state.messages.append({
                    "role": "tool",
                    "content": compact({"tool": tool_name, "ok": result.ok, "content": result.content, "data": result.data})
                })
            except Exception as exc:
                state.consecutive_errors += 1
                state.unchanged_iterations += 1
                state.summary += f"\nException: {exc}"
                state.messages.append({"role": "system", "content": traceback.format_exc()})

        if not state.done:
            state.summary += "\nLoop ended without explicit completion."
        return state


## Wire up registries and default tools

In [ ]:
def build_default_tools(enable_mcp_browser: bool = False) -> ToolRegistry:
    reg = ToolRegistry()
    reg.register(ListDirTool())
    reg.register(ReadFileTool())
    reg.register(WriteFileTool())
    reg.register(SearchTool())
    reg.register(ShellTool())
    reg.register(GitDiffTool())
    reg.register(ScratchpadTool())
    reg.register(VerifyTool())
    if enable_mcp_browser:
        reg.register(BrowserNavigateTool(adapter=StubMCPAdapter()))
    return reg


def build_default_skills() -> SkillRegistry:
    skills = SkillRegistry()
    skills.register(Skill("safe_editing", "Prefer small edits and immediate verification.", "After write_file, call verify or git_diff."))
    skills.register(Skill("repo_orientation", "Inspect structure before editing.", "Start with list_dir/search/read_file before write_file."))
    return skills


## Demo task 1: inspect repository and summarize key notebook files

In [ ]:
REPO_ROOT = "."

task1 = (
    "Inspect the repository and identify key notebook files. "
    "Use tools to gather context and verify with at least one shell or verify command before finishing."
)

# Requires OPENAI_API_KEY in environment.
reasoner = OpenAIReasoner(model="gpt-4.1-mini")
agent = RepoCodingAgent(
    repo_root=REPO_ROOT,
    reasoner=reasoner,
    tools=build_default_tools(enable_mcp_browser=False),
    skills=build_default_skills(),
    max_steps=12,
)

state1 = agent.run(task1)
print(state1.summary)
print(f"\nTrace entries: {len(state1.trace)}")
for t in state1.trace:
    print(f"- step={t.step} phase={t.phase} tool={t.tool_name} ok={t.ok}")


## Demo task 2: create a small artifact and verify it

In [ ]:
task2 = (
    "Create a file at scratch/agent_demo.txt with a 2-line note about this repository, "
    "then verify the file exists and contains the phrase 'repo-aware agent'."
)

state2 = agent.run(task2)
print(state2.summary)
print(f"\nTrace entries: {len(state2.trace)}")
for t in state2.trace:
    print(f"- step={t.step} phase={t.phase} tool={t.tool_name} ok={t.ok}")


## Notes for extracting into a package later

You can move this notebook into a package with:

- `agent/core.py` for state, engine, reasoner
- `agent/tools/*.py` for tool modules
- `agent/skills.py` for skill abstractions
- `agent/mcp.py` for adapter(s)
- `examples/` for scripted tasks

Because this notebook keeps interfaces explicit and decoupled, extraction is mostly copy/move with minimal rewiring.
